In [1]:
import pandas as pd

test_df = pd.read_csv("../clean_data/test_cleaned.csv")

In [2]:
# ===============================
# STEP 1: Imports
# ===============================
import os
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from PIL import Image
from transformers import CLIPProcessor, CLIPModel

# ===============================
# STEP 2: Setup
# ===============================
device = "cuda:1" if torch.cuda.is_available() else "cpu"
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch16").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch16")

# ===============================
# STEP 3: Load Metadata
# ===============================

image_dir = "../images/test"  # folder containing all images
output_dir = "clip_embeddings"
os.makedirs(output_dir, exist_ok=True)

# ===============================
# STEP 4: Function to get embeddings
# ===============================
def get_image_embedding(image_path):

    image = Image.open(image_path).convert("RGB")
    inputs = clip_processor(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        emb = clip_model.get_image_features(**inputs)
    return emb.cpu().numpy().flatten()
   

# ===============================
# STEP 5: Generate Embeddings
# ===============================
from math import ceil

batch_size = 1000  # adjust based on GPU memory
image_embeddings = []

sample_ids = test_df["sample_id"].tolist()
num_batches = ceil(len(sample_ids) / batch_size)

for i in tqdm(range(num_batches), desc="Processing images in batches"):
    batch_ids = sample_ids[i * batch_size : (i + 1) * batch_size]
    
    # Load batch of images
    batch_images = []
    for sid in batch_ids:
        img_path = os.path.join(image_dir, f"{sid}.jpg")
        if not os.path.exists(img_path):
            continue
        image = Image.open(img_path).convert("RGB")
        batch_images.append(image)
    
    # Process batch
    inputs = clip_processor(images=batch_images, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        batch_emb = clip_model.get_image_features(**inputs)
    
    # Move to CPU and convert to numpy
    batch_emb = batch_emb.cpu().numpy()
    image_embeddings.append(batch_emb)

# Stack all embeddings
image_embeddings = np.vstack(image_embeddings)

# Save embeddings
np.save(os.path.join(output_dir, "test_image_embeddings_clip.npy"), image_embeddings)
print("✅ Batched image embeddings saved successfully!")

# Optional: also save as CSV
# pd.DataFrame(image_embeddings).to_csv(os.path.join(output_dir, "image_embeddings.csv"), index=False)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Processing images in batches: 100%|██████████| 75/75 [34:40<00:00, 27.74s/it]

✅ Batched image embeddings saved successfully!


In [4]:
print(image_embeddings.shape)  # (num_images, embedding_dim)

(74999, 512)


In [9]:
# there was 1 missing image, now add a zero embedding for that sample id in that position
for sid in sample_ids:
    img_path = os.path.join(image_dir, f"{sid}.jpg")
    if not os.path.exists(img_path):
        zero_emb = np.zeros((512,), dtype=np.float32)  # CLIP ViT-B/16 has 512-dim embeddings
        image_embeddings_new = np.insert(image_embeddings, sample_ids.index(sid), zero_emb, axis=0)
print(image_embeddings_new.shape)  # (num_images, embedding_dim)
np.save(os.path.join(output_dir, "test_image_embeddings_clip_final.npy"), image_embeddings_new)

(75000, 512)
